# Engenharia de Freatures: Goodreads Dataset
Esta etapa transforma o dataset limpo em uma **estrutura analiticamente granular**, capaz de responder perguntas por gênero literário individual. A principal intervenção é a normalização da coluna `genres`, que no formato original agrupa múltiplos gêneros por registro — impedindo qualquer análise de desempenho por categoria.

- **Escopo:** Desnormalização da coluna `genres` para o modelo um-gênero-por-linha
- **Produto desta etapa:** `goodreads_books_exploded.csv` — estrutura otimizada para análises de popularidade e avaliação por gênero

In [8]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data, save_dataset
from src import estilizar_tabela

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Carregando Dados

In [12]:
caminho = '../data/processed/books/goodreads_books_cleaned.pkl'
df_books = load_data(caminho, tipo_arquivo='pkl')

Dados PKL carregados! Formato: (10965, 12)


---
## Atomização da Coluna `genres`: Habilitando Granularidade Analítica

O dataset limpo armazena múltiplos gêneros por livro em uma única célula delimitada por ponto-e-vírgula — um formato conveniente para armazenamento, mas **incompatível com análises por categoria**. Para responder perguntas como "qual gênero tem a maior média de avaliação?" ou "quais gêneros dominam em número de publicações?", é necessário normalizar a relação livro-gênero para um modelo one-to-many explícito.

- **Decisão arquitetural:** O modelo "explodido" (uma linha por par livro-gênero) é a forma canônica para análises de frequência e performance por categoria em dados de consumo cultural
- **Trade-off aceito:** O dataset resultante terá mais linhas que o original; isso é esperado e desejado — cada linha agora representa uma relação analítica atômica

In [13]:
# Transformando a string contínua em uma lista do Python
df_books['genres'] = df_books['genres'].str.split(';')

# Explodindo a coluna de gêneros para criar uma linha por gênero
df_exploded = df_books.explode('genres')

# Limpando os espaços em branco dos gêneros
df_exploded['genres'] = df_exploded['genres'].str.strip()

# Resetando o índice para evitar índices duplicados
df_exploded.reset_index(drop=True, inplace=True)


In [14]:
# Validação da granularidade atômica: uma linha por par livro-gênero
display(estilizar_tabela(
    df=df_exploded,
    colunas_selecionadas=['Title', 'genres'],
    qtd_linhas=20,
    caption="DataFrame Explodido"
))

,genres
0,Fantasy
1,Young Adult
2,Fiction
3,"Fantasy,Magic"
4,Childrens
5,Adventure
6,Audiobook
7,"Childrens,Middle Grade"
8,Classics
9,Science Fiction Fantasy


In [15]:
df_exploded.info()

<class 'pandas.DataFrame'>
RangeIndex: 94530 entries, 0 to 94529
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   book_id             94530 non-null  int64         
 1   title               94530 non-null  str           
 2   Author              94530 non-null  str           
 3   average_rating      94530 non-null  float64       
 4   isbn                94530 non-null  str           
 5   language_code       94530 non-null  str           
 6   num_pages           94530 non-null  int64         
 7   ratings_count       94530 non-null  int64         
 8   text_reviews_count  94530 non-null  int64         
 9   publication_date    94510 non-null  datetime64[us]
 10  publisher           94530 non-null  str           
 11  genres              94530 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(6)
memory usage: 17.8 MB


---
## Persistência do Dataset Desnormalizado

O DataFrame explodido é salvo como um artefato independente para preservar o dataset enriquecido sem sobrescrever o arquivo limpo da etapa anterior. Essa separação respeita a **imutabilidade das camadas do pipeline**: `cleaned` permanece como fonte de verdade para transformações alternativas, enquanto `exploded` serve como insumo direto para os notebooks de análise.

In [16]:
# Salvando DataFrame explodido para uso futuro
save_dataset(
    df=df_exploded,
    pasta='../data/processed/books',
    nome_arquivo='goodreads_books_exploded', 
    tipo_arquivo='parquet'
)

Sucesso! Ficheiro guardado em '..\data\processed\books\goodreads_books_exploded.parquet'


---
## Conclusão da Engenharia de Features

Nesta etapa, o dataset previamente limpo foi transformado e modelado para suportar consultas analíticas avançadas e extração de *insights* precisos. As principais refatorações incluíram:

* **Atomização Categórica (Explode):** A coluna `genres` foi fatiada e desmembrada, alterando a granularidade do dataset. Agora, o modelo suporta relações N:M, viabilizando análises estatísticas isoladas para cada gênero literário sem o mascaramento de dados agrupados.
* **Inteligência Temporal:** Estruturação de dados de data (`datetime`) para permitir futuras análises de tendências de publicação e engajamento ao longo do tempo.

**Exportação Estratégica:**
O dataset refatorado foi exportado com sucesso para:
  -  `data/processed/goodreads_genres_exploded.csv`  
  -  `data/processed/goodreads_genres_exploded.pkl`


> **Nota Arquitetural:** Para a próxima fase, utilizaremos uma abordagem de *Dual-Dataset*. O arquivo recém-exportado será dedicado exclusivamente às análises de gênero, enquanto o arquivo `goodreads_cleaned.csv` será mantido para métricas de volume geral (evitando a contagem duplicada de obras).

**Próximo Passo:**
Com as bases prontas e a arquitetura definida, o projeto avança para a **Análise Exploratória de Dados (EDA)** para investigar os direcionadores de notas e popularidade das obras.

---